# ⚙️ WMX R2 System Startup

This notebook executes the 5-step standard startup protocol to bring the physical hardware motors into a ready-to-run state (Servo On).  
*Prerequisite: `ros2 launch wmx_r2_package wmx_r2_general_nodes.launch.py` must be running in a background terminal.*

In [1]:
import rclpy
from rclpy.node import Node
from std_srvs.srv import SetBool
from wmx_r2_message.srv import SetEngine, LoadWmxParams, SetAxis

if not rclpy.ok():
    rclpy.init()

class WmxClient(Node):
    def __init__(self):
        super().__init__('wmx_jupyter_client')

    def call(self, srv_type, srv_name, request):
        client = self.create_client(srv_type, srv_name)
        while not client.wait_for_service(timeout_sec=1.0):
            print(f"'{srv_name}' Waiting for '{srv_name}' service server...")
        future = client.call_async(request)
        rclpy.spin_until_future_complete(self, future)
        return future.result()

wmx = WmxClient()
print("✅ ROS2 API Ready.")

✅ ROS2 API Ready.


### Step 1: Create WMX3 Device
*   Calls the **/wmx/engine/set_device** service to allocate a real-time engine interface instance.

In [2]:
req = SetEngine.Request()
req.data = True
req.path = '/opt/wmx3/'
req.name = 'my_device'

res = wmx.call(SetEngine, '/wmx/engine/set_device', req)
if res is not None:
    print("✅ [SUCCESS] WMX3 Device instance activated.")
else:
    print("❌ [ERROR] Failed to create device")

✅ [SUCCESS] WMX3 Device instance activated.


### Step 2: Start EtherCAT Communication
*   Calls the **/wmx/engine/set_comm** service to activate the physical port and initiate the slave motor communication network.

In [3]:
req = SetBool.Request()
req.data = True

res = wmx.call(SetBool, '/wmx/engine/set_comm', req)
if res is not None and res.success:
    print(f"✅ [SUCCESS] EtherCAT communication started: {res.message}")
else:
    print("❌ [ERROR] Failed to start communication.")

✅ [SUCCESS] EtherCAT communication started: Communication started


### Step 3: Load WMX Parameter XML File
*   Calls the **/wmx/params/load** service to inject the target device (Diffbot) gear ratios and limit settings.

In [4]:
import os

xml_path = os.path.expanduser('~/workspaces/movensys_ws/install/wmx_r2_package/share/wmx_r2_package/config/diffbot_wmx_parameters.xml')

req = LoadWmxParams.Request()
req.file_path = xml_path

res = wmx.call(LoadWmxParams, '/wmx/params/load', req)
if res is not None:
    print(f"✅ XML parameters injected: {xml_path}")
else:
    print("❌ [ERROR] Failed to load parameter XML file.")

✅ XML parameters injected: /home/mic-733ao/workspaces/movensys_ws/install/wmx_r2_package/share/wmx_r2_package/config/diffbot_wmx_parameters.xml


### Step 4: Clear Axix Alarms
*   Forces the removal of error latches for servo drives 0 and 1 via the **/wmx/axis/clear_alarm** service.

In [5]:
req = SetAxis.Request()
req.index = [0]
req.data = [0]

res = wmx.call(SetAxis, '/wmx/axis/clear_alarm', req)
if res is not None:
    print("✅ [SUCCESS] Sent alarm clear command for Axis 0.")
else:
    print("❌ [ERROR] Failed to request alarm clear.")

✅ [SUCCESS] Sent alarm clear command for Axis 0.


### Step 5: Motor Servo On
*   Calls the **/wmx/axis/set_on** service to apply power to the physical axis and release the brake, holding the hardware in a ready-to-control state.

In [8]:
req = SetAxis.Request()
req.index = [0]
req.data = [0]

res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
if res is not None:
    print("⚡━━━━━━━ SYSTEM STARTUP SUCCESS ━━━━━━━⚡")
    print("🚀 [SUCCESS] Synchronous servo startup completed for Motor 0.")
else:
    print("❌ [ERROR] Failed to enable servo (Servo On).")

⚡━━━━━━━ SYSTEM STARTUP SUCCESS ━━━━━━━⚡
🚀 [SUCCESS] Synchronous servo startup completed for Motor 0.
